In [21]:
# Nettoyage complet
print("Nettoyage...")
%reset -f

%whos

Nettoyage...
Interactive namespace is empty.


In [22]:
# Import de tous les packages dont on va avoir besoin

import numpy.random as rd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import pandas as pd 
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sqlalchemy import text
import sqlalchemy
import gc
import pyodbc
import os
from datetime import datetime
from pandas import NaT
import math
from scipy.stats import expon
import geopandas as gpd
from shapely.geometry import Point
import contextily as ctx
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
import time
from scipy.stats import gaussian_kde
from sklearn.neighbors import KernelDensity
from scipy.stats import gaussian_kde
import pickle
from joblib import Parallel, delayed
from matplotlib.colors import LogNorm
from matplotlib.colors import PowerNorm 
from shapely.geometry import Point, Polygon
from shapely.ops import unary_union, polygonize
from scipy.spatial import Voronoi
from shapely.geometry import box
from shapely.ops import unary_union
from geopy.distance import geodesic
from geovoronoi import voronoi_regions_from_coords
import requests
import geodatasets
import cartopy
import cartopy.io.shapereader as shpreader
import concurrent.futures
import psutil

In [23]:
requete = """ 
WITH villes_agg AS (
    SELECT 
        a.adr_adresse_ville,
        COUNT(*) AS nb_lots_ville,
        NTILE(100) OVER (ORDER BY COUNT(*) DESC) AS quantile_nb_lots_ville
    FROM dbo.tj_offre_lot tj
    JOIN dbo.t_lot l ON tj.lot_id = l.lot_id
    JOIN dbo.t_adresse a ON l.adresse_id = a.adresse_id
    WHERE 
        l.lot_actif = 1 
        AND a.pays_id = 28 
        AND a.adr_plan_x IS NOT NULL 
        AND a.adr_plan_y IS NOT NULL 
        AND CAST(a.adr_plan_x AS FLOAT) BETWEEN -5.5 AND 9.5 
        AND CAST(a.adr_plan_y AS FLOAT) BETWEEN 41.0 AND 51.5
    GROUP BY a.adr_adresse_ville),
villes_sample AS (
    SELECT 
        a.adr_adresse_ville,
        MAX(a.adr_adresse) AS adr_adresse,  -- ou NEWID() dans ROW_NUMBER pour du random
        CAST(MAX(a.adr_plan_x) AS FLOAT) AS longitude,
        CAST(MAX(a.adr_plan_y) AS FLOAT) AS latitude
    FROM dbo.tj_offre_lot tj
    JOIN dbo.t_lot l ON tj.lot_id = l.lot_id
    JOIN dbo.t_adresse a ON l.adresse_id = a.adresse_id
    WHERE 
        l.lot_actif = 1 
        AND a.pays_id = 28 
        AND a.adr_plan_x IS NOT NULL 
        AND a.adr_plan_y IS NOT NULL 
        AND CAST(a.adr_plan_x AS FLOAT) BETWEEN -5.5 AND 9.5 
        AND CAST(a.adr_plan_y AS FLOAT) BETWEEN 41.0 AND 51.5
    GROUP BY a.adr_adresse_ville
    )
SELECT 
    s.adr_adresse_ville,
    s.adr_adresse,
    s.longitude,
    s.latitude,
    v.nb_lots_ville,
    v.quantile_nb_lots_ville
FROM villes_sample s
JOIN villes_agg v ON s.adr_adresse_ville = v.adr_adresse_ville
ORDER BY nb_lots_ville DESC
"""

In [24]:
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=172.16.11.34;"
    "DATABASE=LogiProDev;"
    "UID=quentin;"
    "PWD={Barbidur1;SQL}"
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

data = pd.read_sql_query(requete, conn)

conn.commit()
cursor.close()
conn.close()

# Coordonnées de Paris
lat_paris = 48.856613
lon_paris = 2.352222

# Remplacer les valeurs pour Paris
data.loc[data["adr_adresse_ville"] == "PARIS", ["latitude", "longitude"]] = [lat_paris, lon_paris]

data = data.sort_values(
    by="quantile_nb_lots_ville",
    ascending=False
).reset_index(drop=True)  

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=172.16.11.34;"
    "DATABASE=Superset;"
    "UID=quentin;"
    "PWD={Barbidur1;SQL}"
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("TRUNCATE TABLE Superset.db_owner.donnees_pour_carte_lots")

insert_sql = """
INSERT INTO Superset.db_owner.donnees_pour_carte_lots (
    adr_adresse_ville,
    adr_adresse,
    latitude,
    longitude,
    nb_lots_ville,
    quantile_nb_lots_ville
)
VALUES (?, ?, ?, ?, ?, ?)
"""

cursor.fast_executemany = True

data = list(
    data[[
        "adr_adresse_ville",
        "adr_adresse",
        "latitude",
        "longitude",
        "nb_lots_ville",
        "quantile_nb_lots_ville"
    ]].itertuples(index=False, name=None)
)

cursor.executemany(insert_sql, data)

conn.commit()
cursor.close()
conn.close()

C:\Users\quent\AppData\Local\Temp\ipykernel_20284\2090386195.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql_query(requete, conn)
